# Direction 2 · Phase C · Step 6: Pre-Layer4 Mechanism-Derived Repair & 6-Arm Ablation (Kaggle GPU)

**Settings:** Enable **GPU T4 x1** and **Internet ON**.

**Zero manual uploads required:**
- Clones `spurious_correlation_mechanisms` from GitHub.
- Auto-downloads the 5 saved `standard_erm` ResNet-50 `ckpt_last.pt` backbones, Waterbirds images (~490 MB), and Caltech `CUB_200_2011` pixel masks (~39 MB, used **strictly for post-hoc counterfactual & gate-alignment evaluation, never during training**).
- Runs 6 matched-budget arms on cleaned 5-fold validation CV (`split == 1`, zero test access):
  1. `affine_dfr_linear` (Step 5 bar to beat: `92.69%` OOF-Thr WGA / `98.02%` Min-AUC)
  2. `affine_dfr_cosine` (spherical readout control)
  3. `pre_l4_gate_only` (`layer3.5` $14\times 14$ spatial gate alone vs. `DAR` at `layer4.2`)
  4. `pre_l4_gate_bn_ce` (`layer3.5` gate + `bn_all` + cosine, supervised CE only)
  5. `pre_l4_repair_bn_l3l4` (full mechanism-derived repair, `bn_l3_l4` + `layer3.5` context-swap invariance)
  6. `pre_l4_repair_bn_all` (full mechanism-derived repair, `bn_all` + `layer3.5` context-swap invariance)
- Outputs `/kaggle/working/rep_repair_phase_c_step6.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/kaggle/working/phase_c_assets')
OUT_DIR = Path('/kaggle/working/rep_repair_phase_c_step6')

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Pre-flight unit tests.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_phase_c_step6.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_phase_c_step6',
    '--work-dir', str(WORK_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'step6_repair_ablation.png')))
display(Image(filename=str(OUT_DIR / 'gate_maps_visual.png')))

zip_path = Path('/kaggle/working/rep_repair_phase_c_step6.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved results bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')